# all-MiniLM-L6-v2 - Fast Lightweight Text Embeddings on Amazon SageMaker

Deploys [sentence-transformers/all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) from AWS
Marketplace as a SageMaker endpoint inside **your own AWS account**. Your text never
leaves your VPC and there are no external API calls or token limits.

**Model facts** (from the official model card): 384-dimensional vectors, 256 max input
tokens, mean pooling, Apache-2.0 licence.

Vectors are mean-pooled and L2-normalised, so cosine similarity is a plain dot product.

**When to choose all-MiniLM-L6-v2:**
- Lightweight semantic search, clustering, and sentence-level similarity
- Latency-sensitive applications needing sub-millisecond CPU inference
- Cost-efficient at **$0.07/hr** with strong English sentence understanding

## 1. Prerequisites

1. Subscribe to the product in AWS Marketplace.
2. Copy the **model package ARN** shown on the product's launch page for your Region.
3. Run this notebook with a role that has `AmazonSageMakerFullAccess`.

In [ ]:
!pip install -qU sagemaker boto3

In [ ]:
import json

import boto3
import sagemaker
from sagemaker import ModelPackage

# Paste the model package ARN from the product's launch page for YOUR Region.
MODEL_PACKAGE_ARN = "<paste-model-package-arn-here>"

INSTANCE_TYPE = "ml.m5.xlarge"  # the recommended real-time instance
ENDPOINT_NAME = "all-minilm-l6-v2"
MAX_SEQ_LENGTH = 256

session = sagemaker.Session()
try:
    role = sagemaker.get_execution_role()
except ValueError:
    # Running outside SageMaker -- specify your role ARN explicitly
    role = "arn:aws:iam::<ACCOUNT-ID>:role/<SAGEMAKER-ROLE-NAME>"
print("Region:", session.boto_region_name)
print("region:", session.boto_region_name)

## 2. Deploy a real-time endpoint

Takes roughly 6-9 minutes. The endpoint bills **$0.07/hr** while it exists,
so do not skip section 5.

In [ ]:
model = ModelPackage(
    role=role,
    model_package_arn=MODEL_PACKAGE_ARN,
    sagemaker_session=session,
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
)
print("endpoint ready:", ENDPOINT_NAME)

## 3. Embed text

The endpoint accepts `application/json` shaped `{"inputs": "..."}` for a single string,
or `{"inputs": ["...", "..."]}` for a batch. It returns
`{"embeddings": [[...]], "dim": 384}`.

Input is truncated to `MAX_SEQ_LENGTH=256` tokens. Sentences longer than ~190 words
will be silently truncated.

In [ ]:
runtime = boto3.client("sagemaker-runtime")
def embed(texts):
    """Return a list of 384-dimension mean-pooled L2-normalised embedding vectors."""
    if isinstance(texts, str):
        texts = [texts]
    try:
    try:
        response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps({"inputs": texts}),
        )
        result = json.loads(response["Body"].read())
    except Exception as e:
        print(f"Error invoking endpoint: {e}")
        raise
    if "embeddings" not in result:
        raise ValueError(f"Unexpected response format: {result}")
    return result["embeddings"]
sample_sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "A fast auburn fox leaps above a sleepy canine.",
    "Amazon SageMaker makes deploying ML models straightforward.",
    "The weather in Seattle is often cloudy in November.",
]
vectors = embed(sample_sentences)
print("dimensions:", len(vectors[0]))  # should be 384
print("sentences embedded:", len(vectors))
print("first 8 values of sentence 0:", [round(v, 5) for v in vectors[0][:8]])

## 4. Cosine similarity

Vectors are already L2-normalised, so cosine similarity is a plain dot product.
Sentences with similar meaning score near 1.0; unrelated sentences score near 0.

In [ ]:
def cosine_similarity(a, b):
    """Dot product of two L2-normalised vectors equals cosine similarity."""
    return sum(x * y for x, y in zip(a, b))


print("Pairwise cosine similarities:\n")
for i, sent_i in enumerate(sample_sentences):
    for j, sent_j in enumerate(sample_sentences):
        if j <= i:
            continue
        sim = cosine_similarity(vectors[i], vectors[j])
        print(f"  [{i}] vs [{j}]: {sim:.4f}")
        print(f"       '{sent_i[:60]}...' " if len(sent_i) > 60 else f"       '{sent_i}'")
        print(f"       '{sent_j[:60]}...' " if len(sent_j) > 60 else f"       '{sent_j}'")
        print()

## 5. Clean up

Delete the endpoint when you are done. It bills $0.07/hr for as long as it is running.

In [ ]:
predictor.delete_model()
predictor.delete_endpoint()
print("deleted:", ENDPOINT_NAME)